# CIFAR Lens 91 percent target

This notebook trains a new ResNet20 candidate from scratch. **91% is a target, not a result.** Select Runtime → Change runtime type → T4 GPU and runtime version 2026.01 (Python 3.12) before running. The official test set is held out until final evaluation.

Original project: Rory O'Brien, Amir Hamza and MD Suyaib Rahman. This portfolio extension, notebook and new training recipe were developed with OpenAI Codex assistance.


In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), "Select runtime version 2026.01 for Python 3.12."
import subprocess, os
from pathlib import Path
repo = Path("/content/cifar-lens")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amirhamzaaaa4/cifar-lens.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["python", "-m", "pip", "install", "tensorflow==2.19.0", "keras==3.10.0", "numpy==2.1.3", "scikit-learn>=1.6,<2", "Pillow>=11,<13"], check=True)


## Verify GPU availability
If the check fails, select a GPU runtime and restart the notebook runtime. This notebook does not buy compute or upgrade a Google account. GPU availability is controlled by Colab.


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
assert gpus, "Select a GPU runtime before starting the 200-epoch training run."
print("TensorFlow:", tf.__version__, "GPUs:", gpus)


## Train using an independent validation split
The run uses 45,000 training and 5,000 validation images. Checkpoints are selected using validation loss. A cosine schedule reduces the learning rate across 200 epochs. Do not stop the run because a test score looks favorable.

For protection against disconnects, an optional copy of checkpoints to Drive is available below. Mounting Drive is optional and requires your Google consent.


In [ ]:
# Optional: uncomment these lines to persist checkpoints in your Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# output = Path("/content/drive/MyDrive/CIFAR-Lens/resnet20-target91")
output = Path("artifacts/resnet20-target91")


In [ ]:
import sys
command = [sys.executable, "-m", "lens.train", "--architecture", "resnet20", "--epochs", "200", "--batch-size", "128", "--target-accuracy", "0.91", "--output", str(output)]
if (output / "latest.keras").exists():
    command.append("--resume")
subprocess.run(command, check=True)


## Actual held-out test result
The report records whether the target was achieved. A result below 91% must be reported honestly; it is not rounded up or replaced with validation accuracy.


In [ ]:
import json
results = json.loads((output / "results.json").read_text())
print(f"Test accuracy: {results["accuracy"]:.2%}")
print(f"Macro F1: {results["macro_f1"]:.4f}")
print("Target met:", results["target_met"])


## Download the verified experiment
The zip contains actual model weights, metrics, error examples and sample images. It excludes intermediate checkpoints. These are the files needed by the demo app.


In [ ]:
import zipfile
archive = Path("/content/cifar-lens-resnet20-trained.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for item in output.rglob("*"):
        if item.is_file() and item.name not in ["best.keras", "latest.keras", "epochs.csv", "progress.json"]:
            z.write(item, Path("artifacts/resnet20-target91") / item.relative_to(output))
from google.colab import files
files.download(str(archive))
